# 04 · Controlled atlas comparison: dk68 vs lobes12 vs r18 (development CV only; exploratory)

`configs/final/roi_comparison.yaml`.

- **dk68**: reference (the validated 68-ROI stores)
- **lobes12**: rule-recommended (exact-BEM atlas_resolution rule)
- **r18**: exploratory (post-result author approval, amendment A1; fails the LCMV own-region dominance criterion)

The same 52 development participants, the same five participant-disjoint folds (`xu_dev_cv`), window mode
B, the same training procedures and seeds, LCMV and wMNE, for:

- `exp1_source_psd_de`: `logreg__psd`, `svm__psd`, `logreg__de`, `svm__de`
- `exp4_band_mean`: `plv__psd`
- `exp5_multibranch`: `plv__separate__mean`

GNNs on dense weighted PLV, exactly 60 epochs. **The holdout is not evaluated** (the code refuses it until the final
analysis is frozen). Paired differences on the same participants: `lobes12 - dk68`, `r18 - dk68`, `r18 - lobes12`.

The report keeps **source localization / spatial resolution** (label-free; notebook 02), **classification per
atlas** (participant-level pooled out-of-fold ROC-AUC with DeLong CI, balanced accuracy, F1, sensitivity,
specificity) and **paired differences** (paired DeLong, participant-bootstrap CIs) in separate sections. All
comparisons are exploratory and not corrected for multiplicity; a higher AUC alone does not make an atlas better,
and the summary never forces a winner. **This notebook stops after the report: no atlas is selected** (no
`decisions/atlas.json` is written); the next stage is your decision.

The dk68 runs are the same units the experiment notebooks reuse (nothing is computed twice). An atlas whose stores
are not authorized / built (e.g. r18 without the recorded approval) is skipped and its comparisons are reported as
incomplete. GPU runtime (T4). Resumable.

**How to use:** *Runtime → Run all*. Allow Google Drive access when asked. If Colab disconnects, open the notebook
again and *Run all* again: finished work is kept on Google Drive and skipped. Everything is written to
`<outputs>/FINAL_v1/` on Drive (never to the temporary Colab disk); the validated graph stores and all earlier
results are only read.

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only to move the release code pin deliberately to the newest commit
                      # (<outputs>/FINAL_v1/CODE_COMMIT.txt; unfinished work is then recomputed).
BRANCH = 'claude/unified-final-framework'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
FINAL = OUT + '/FINAL_v1'
print('release outputs on Drive:', FINAL)

In [ ]:
import glob, json
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_rows', 400, 'display.width', 250, 'display.max_columns', 40)

def latest(pattern):
    found = sorted(glob.glob(pattern))
    return found[-1] if found else None

## 1. Stores that take part (read-only verification)

In [ ]:
ATLASES = []
for atlas in ["dk68", "lobes12", "r18"]:
    try:
        sh(f"python scripts/final.py stores --atlas {atlas}")
        ATLASES.append(atlas)
    except RuntimeError:
        if atlas == 'dk68':
            raise
        print(f'{atlas}: stores missing or not verified: skipped (its comparisons will be incomplete)')
print('atlases compared:', ATLASES)

## 2. Runs (development CV only; each unit is checkpointed)

In [ ]:
RUNS = [["exp1", "logreg__psd", "dk68", "lcmv", "none"], ["exp1", "svm__psd", "dk68", "lcmv", "none"], ["exp1", "logreg__de", "dk68", "lcmv", "none"], ["exp1", "svm__de", "dk68", "lcmv", "none"], ["exp4", "plv__psd", "dk68", "lcmv", "dense"], ["exp5", "plv__separate__mean", "dk68", "lcmv", "dense"], ["exp1", "logreg__psd", "dk68", "wmne", "none"], ["exp1", "svm__psd", "dk68", "wmne", "none"], ["exp1", "logreg__de", "dk68", "wmne", "none"], ["exp1", "svm__de", "dk68", "wmne", "none"], ["exp4", "plv__psd", "dk68", "wmne", "dense"], ["exp5", "plv__separate__mean", "dk68", "wmne", "dense"], ["exp1", "logreg__psd", "lobes12", "lcmv", "none"], ["exp1", "svm__psd", "lobes12", "lcmv", "none"], ["exp1", "logreg__de", "lobes12", "lcmv", "none"], ["exp1", "svm__de", "lobes12", "lcmv", "none"], ["exp4", "plv__psd", "lobes12", "lcmv", "dense"], ["exp5", "plv__separate__mean", "lobes12", "lcmv", "dense"], ["exp1", "logreg__psd", "lobes12", "wmne", "none"], ["exp1", "svm__psd", "lobes12", "wmne", "none"], ["exp1", "logreg__de", "lobes12", "wmne", "none"], ["exp1", "svm__de", "lobes12", "wmne", "none"], ["exp4", "plv__psd", "lobes12", "wmne", "dense"], ["exp5", "plv__separate__mean", "lobes12", "wmne", "dense"], ["exp1", "logreg__psd", "r18", "lcmv", "none"], ["exp1", "svm__psd", "r18", "lcmv", "none"], ["exp1", "logreg__de", "r18", "lcmv", "none"], ["exp1", "svm__de", "r18", "lcmv", "none"], ["exp4", "plv__psd", "r18", "lcmv", "dense"], ["exp5", "plv__separate__mean", "r18", "lcmv", "dense"], ["exp1", "logreg__psd", "r18", "wmne", "none"], ["exp1", "svm__psd", "r18", "wmne", "none"], ["exp1", "logreg__de", "r18", "wmne", "none"], ["exp1", "svm__de", "r18", "wmne", "none"], ["exp4", "plv__psd", "r18", "wmne", "dense"], ["exp5", "plv__separate__mean", "r18", "wmne", "dense"]]      # (experiment, variant, atlas, inverse, graph)
ERRORS = []
for exp, variant, atlas, inverse, graph in RUNS:
    if atlas not in ATLASES:
        continue
    g = '' if graph == 'none' else f' --graph {graph}'
    try:
        sh(f"python scripts/final.py run --exp {exp} --inverse {inverse} --mode B --atlas {atlas}{g} "
           f"--variants {variant}")
    except RuntimeError:
        ERRORS.append((exp, variant, atlas, inverse))
if ERRORS:
    raise RuntimeError(f'units did not complete: {ERRORS}; completed units are saved. Run all again.')

## 3. The comparison (saved outputs only)

In [ ]:
try:
    sh("python scripts/final.py roi-compare")
except RuntimeError:
    print('some comparisons are incomplete (see above); the complete ones are reported')
d = latest(FINAL + '/roi_comparison/compare_*')
from IPython.display import Markdown
if d:
    display(Markdown(open(d + '/report.md').read()))

## Stop here

The comparison is complete for review. Nothing has been selected: no analysis atlas is recorded, notebook 03 and the
experiment notebooks are not started. The next stage is the author's explicit decision (FINAL_RUN_GUIDE.md §3.1).